# 03 — Enhanced Gold Layer: Business Aggregates & ML Features

**What this notebook does:**
- Reads `silver_mydata` Delta table
- Creates multiple business-level Gold tables:
  1. `gold_area_crime_summary` — crime counts, weapon rates, victim demographics per LAPD area
  2. `gold_monthly_trends` — month-over-month crime trends with YoY context
  3. `gold_hourly_patterns` — crime distribution by hour and day type (weekday vs weekend)
  4. `gold_victim_profiles` — victim demographic breakdowns by crime type
  5. `gold_weapon_area_analysis` — weapon usage patterns across areas and crime types
  6. `gold_ml_features` — feature table ready for MLlib model training

**Prerequisite:** Run `02_silver_layer.ipynb` first.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df = spark.read.table("silver_mydata")
total = df.count()
print(f"Silver rows loaded: {total:,}")
print(f"Columns: {len(df.columns)}")
df.printSchema()

Silver rows loaded: 62,105
Columns: 39
root
 |-- DR_NO: integer (nullable = true)
 |-- Date_Rptd: date (nullable = true)
 |-- DATE_OCC: date (nullable = true)
 |-- TIME_OCC: integer (nullable = true)
 |-- AREA: integer (nullable = true)
 |-- AREA_NAME: string (nullable = true)
 |-- Rpt_Dist_No: integer (nullable = true)
 |-- Part_1-2: integer (nullable = true)
 |-- Crm_Cd: integer (nullable = true)
 |-- Crm_Cd_Desc: string (nullable = true)
 |-- Mocodes: string (nullable = true)
 |-- Vict_Age: integer (nullable = true)
 |-- Vict_Sex: string (nullable = true)
 |-- Vict_Descent: string (nullable = true)
 |-- Premis_Cd: double (nullable = true)
 |-- Premis_Desc: string (nullable = true)
 |-- Weapon_Used_Cd: double (nullable = true)
 |-- Weapon_Desc: string (nullable = true)
 |-- Status: string (nullable = true)
 |-- Status_Desc: string (nullable = true)
 |-- Crm_Cd_1: double (nullable = true)
 |-- Crm_Cd_2: double (nullable = true)
 |-- Crm_Cd_3: double (nullable = true)
 |-- Crm_Cd_4: st

# ─────────────────────────────────────────────────────────────────────
# Gold Table 1: Area-Level Crime Summary
# ─────────────────────────────────────────────────────────────────────

In [0]:
area_summary = (
    df.groupBy("AREA_NAME")
    .agg(
        F.count("*").alias("Total_Crimes"),
        F.sum("Has_Weapon").alias("Weapon_Crimes"),
        F.round(F.avg("Vict_Age_Clean"), 1).alias("Avg_Victim_Age"),
        F.round(F.avg("Reporting_Delay"), 1).alias("Avg_Reporting_Delay"),
        F.countDistinct("Crm_Cd_Desc").alias("Unique_Crime_Types"),
        F.sum(F.when(F.col("Vict_Sex_Clean") == "Male", 1).otherwise(0)).alias("Male_Victims"),
        F.sum(F.when(F.col("Vict_Sex_Clean") == "Female", 1).otherwise(0)).alias("Female_Victims"),
        F.sum(F.when(F.col("Hour").between(18, 23), 1).otherwise(0)).alias("Evening_Crimes"),
        F.sum(F.when(F.col("IsWeekend") == 1, 1).otherwise(0)).alias("Weekend_Crimes"),
    )
    .withColumn("Weapon_Rate_Pct", F.round(F.col("Weapon_Crimes") / F.col("Total_Crimes") * 100, 2))
    .withColumn("Evening_Pct", F.round(F.col("Evening_Crimes") / F.col("Total_Crimes") * 100, 2))
    .withColumn("Weekend_Pct", F.round(F.col("Weekend_Crimes") / F.col("Total_Crimes") * 100, 2))
    .orderBy(F.desc("Total_Crimes"))
)

area_summary.write.format("delta").mode("overwrite").saveAsTable("gold_area_crime_summary")
print("✓ gold_area_crime_summary saved")
display(area_summary)

✓ gold_area_crime_summary saved


AREA_NAME,Total_Crimes,Weapon_Crimes,Avg_Victim_Age,Avg_Reporting_Delay,Unique_Crime_Types,Male_Victims,Female_Victims,Evening_Crimes,Weekend_Crimes,Weapon_Rate_Pct,Evening_Pct,Weekend_Pct
Central,6024,172,35.6,3.2,54,2060,1180,1765,1900,2.86,29.3,31.54
Pacific,4481,913,39.9,6.1,49,1485,1285,1510,1283,20.37,33.7,28.63
Southwest,4089,279,31.2,4.3,58,988,931,1287,1239,6.82,31.47,30.3
N Hollywood,3353,163,39.0,4.3,59,1223,766,1104,923,4.86,32.93,27.53
Wilshire,3057,30,36.7,5.6,38,738,820,1083,895,0.98,35.43,29.28
Hollywood,2936,164,37.0,6.1,45,916,937,968,932,5.59,32.97,31.74
Rampart,2920,102,36.8,4.5,43,669,573,919,742,3.49,31.47,25.41
Northeast,2796,45,38.8,6.7,44,798,646,1036,851,1.61,37.05,30.44
Devonshire,2778,126,41.8,5.7,60,693,588,843,719,4.54,30.35,25.88
Newton,2704,205,35.5,3.5,56,1047,512,744,859,7.58,27.51,31.77


# ─────────────────────────────────────────────────────────────────────
# Gold Table 2: Monthly Crime Trends
# ─────────────────────────────────────────────────────────────────────

In [0]:
monthly = (
    df.groupBy("Year", "Month")
    .agg(
        F.count("*").alias("Crime_Count"),
        F.sum("Has_Weapon").alias("Weapon_Crimes"),
        F.round(F.avg("Reporting_Delay"), 2).alias("Avg_Delay"),
        F.countDistinct("AREA_NAME").alias("Areas_Affected"),
    )
    .orderBy("Year", "Month")
)

w = Window.orderBy("Year", "Month")
monthly = (
    monthly
    .withColumn("Prev_Month_Count", F.lag("Crime_Count").over(w))
    .withColumn("MoM_Change_Pct",
                F.round((F.col("Crime_Count") - F.col("Prev_Month_Count"))
                        / F.col("Prev_Month_Count") * 100, 2))
)

monthly.write.format("delta").mode("overwrite").saveAsTable("gold_monthly_trends")
print("✓ gold_monthly_trends saved")
display(monthly)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


✓ gold_monthly_trends saved


Year,Month,Crime_Count,Weapon_Crimes,Avg_Delay,Areas_Affected,Prev_Month_Count,MoM_Change_Pct
2024,5,9386,1338,6.16,21,null,null
2024,6,8126,368,6.51,21,9386,-13.42
2024,7,8170,309,5.74,21,8126,0.54
2024,8,8324,338,5.5,21,8170,1.88
2024,9,8309,375,4.85,21,8324,-0.18
2024,10,7817,369,4.19,21,8309,-5.92
2024,11,7179,312,3.42,21,7817,-8.16
2024,12,4697,201,2.33,21,7179,-34.57
2025,1,26,11,1.62,14,4697,-99.45
2025,2,44,11,1.2,15,26,69.23


# ─────────────────────────────────────────────────────────────────────
# Gold Table 3: Hourly Crime Patterns (Weekday vs Weekend)
# ─────────────────────────────────────────────────────────────────────

In [0]:
hourly = (
    df.groupBy("Hour", "IsWeekend")
    .agg(
        F.count("*").alias("Crime_Count"),
        F.sum("Has_Weapon").alias("Weapon_Count"),
        F.round(F.avg("Vict_Age_Clean"), 1).alias("Avg_Victim_Age"),
    )
    .withColumn("Day_Type", F.when(F.col("IsWeekend") == 1, "Weekend").otherwise("Weekday"))
    .orderBy("Hour", "IsWeekend")
)

hourly.write.format("delta").mode("overwrite").saveAsTable("gold_hourly_patterns")
print("✓ gold_hourly_patterns saved")
display(hourly)

✓ gold_hourly_patterns saved


Hour,IsWeekend,Crime_Count,Weapon_Count,Avg_Victim_Age,Day_Type
0,0,1173,96,38.1,Weekday
0,1,752,75,33.9,Weekend
1,0,977,77,37.3,Weekday
1,1,608,46,33.0,Weekend
2,0,819,72,37.5,Weekday
2,1,486,36,34.7,Weekend
3,0,785,48,40.1,Weekday
3,1,391,41,37.1,Weekend
4,0,686,50,39.3,Weekday
4,1,301,31,39.6,Weekend


# ─────────────────────────────────────────────────────────────────────
# Gold Table 4: Victim Demographic Profiles by Crime Type
# ─────────────────────────────────────────────────────────────────────

In [0]:
victim_profiles = (
    df.filter(F.col("Vict_Age_Clean").isNotNull())
    .groupBy("Crm_Cd_Desc")
    .agg(
        F.count("*").alias("Total_Incidents"),
        F.round(F.avg("Vict_Age_Clean"), 1).alias("Avg_Age"),
        F.expr("percentile_approx(Vict_Age_Clean, 0.5)").alias("Median_Age"),
        F.min("Vict_Age_Clean").alias("Min_Age"),
        F.max("Vict_Age_Clean").alias("Max_Age"),
        F.sum(F.when(F.col("Vict_Sex_Clean") == "Male", 1).otherwise(0)).alias("Male_Count"),
        F.sum(F.when(F.col("Vict_Sex_Clean") == "Female", 1).otherwise(0)).alias("Female_Count"),
        F.sum("Has_Weapon").alias("Weapon_Count"),
    )
    .withColumn("Male_Pct", F.round(F.col("Male_Count") / F.col("Total_Incidents") * 100, 1))
    .withColumn("Female_Pct", F.round(F.col("Female_Count") / F.col("Total_Incidents") * 100, 1))
    .filter(F.col("Total_Incidents") > 50)
    .orderBy(F.desc("Total_Incidents"))
)

victim_profiles.write.format("delta").mode("overwrite").saveAsTable("gold_victim_profiles")
print("✓ gold_victim_profiles saved")
display(victim_profiles)

✓ gold_victim_profiles saved


Crm_Cd_Desc,Total_Incidents,Avg_Age,Median_Age,Min_Age,Max_Age,Male_Count,Female_Count,Weapon_Count,Male_Pct,Female_Pct
BURGLARY FROM VEHICLE,5173,37.4,34,3,99,2858,2270,193,55.2,43.9
THEFT PLAIN - PETTY ($950 & UNDER),4947,38.0,35,2,99,2406,2443,208,48.6,49.4
"VANDALISM - FELONY ($400 & OVER, ALL CHURCH VANDALISMS)",3343,40.3,37,16,99,1705,1579,125,51.0,47.2
THEFT OF IDENTITY,3215,42.0,39,17,90,1452,1741,2,45.2,54.2
THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND OVER),3195,41.0,37,2,99,1785,1393,51,55.9,43.6
"THEFT-GRAND ($950.01 & OVER)EXCPT,GUNS,FOWL,LIVESTK,PROD",2478,39.0,36,11,99,1319,1085,264,53.2,43.8
SHOPLIFTING - PETTY THEFT ($950 & UNDER),1440,32.0,27,14,90,838,238,0,58.2,16.5
THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),973,39.4,35,18,85,547,424,14,56.2,43.6
PICKPOCKET,895,33.7,29,10,99,213,677,0,23.8,75.6
VANDALISM - MISDEAMEANOR ($399 OR UNDER),892,40.9,37,18,84,444,425,41,49.8,47.6


# ─────────────────────────────────────────────────────────────────────
# Gold Table 5: Weapon Usage by Area and Crime Type
# ─────────────────────────────────────────────────────────────────────

In [0]:
weapon_analysis = (
    df.filter(F.col("Has_Weapon") == 1)
    .groupBy("AREA_NAME", "Crm_Cd_Desc")
    .agg(
        F.count("*").alias("Weapon_Incidents"),
        F.round(F.avg("Hour"), 1).alias("Avg_Hour"),
        F.round(F.avg("Vict_Age_Clean"), 1).alias("Avg_Victim_Age"),
    )
    .filter(F.col("Weapon_Incidents") >= 5)
    .orderBy(F.desc("Weapon_Incidents"))
)

weapon_analysis.write.format("delta").mode("overwrite").saveAsTable("gold_weapon_area_analysis")
print("✓ gold_weapon_area_analysis saved")
display(weapon_analysis.limit(20))

✓ gold_weapon_area_analysis saved


AREA_NAME,Crm_Cd_Desc,Weapon_Incidents,Avg_Hour,Avg_Victim_Age
Pacific,"THEFT-GRAND ($950.01 & OVER)EXCPT,GUNS,FOWL,LIVESTK,PROD",262,13.6,42.5
Pacific,THEFT PLAIN - PETTY ($950 & UNDER),211,13.2,43.2
Pacific,BURGLARY FROM VEHICLE,92,13.0,39.8
Pacific,BATTERY - SIMPLE ASSAULT,85,14.5,42.2
77th Street,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",73,12.2,37.5
Newton,BURGLARY FROM VEHICLE,69,11.6,32.4
Southeast,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",64,13.2,33.7
Southwest,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",63,13.5,32.7
Pacific,"VANDALISM - FELONY ($400 & OVER, ALL CHURCH VANDALISMS)",63,9.5,42.8
Southeast,ROBBERY,62,13.5,37.7


# ─────────────────────────────────────────────────────────────────────
# Gold Table 6: ML-Ready Feature Table
# Purpose: Pre-computed features for MLlib model training
# ─────────────────────────────────────────────────────────────────────

In [0]:
ml_features = (
    df.select(
        "AREA", "AREA_NAME", "Hour", "Month", "IsWeekend",
        "Reporting_Delay", "Has_Weapon",
        "Vict_Age_Clean", "Vict_Sex_Clean",
        "Crm_Cd_Desc", "Premis_Desc",
    )
    .filter(F.col("Crm_Cd_Desc").isNotNull())
    .fillna({"Vict_Age_Clean": 0, "Vict_Sex_Clean": "Unknown", "Premis_Desc": "Unknown"})

    .withColumn("TimeBucket",
        F.when(F.col("Hour").between(6, 11), "Morning")
         .when(F.col("Hour").between(12, 17), "Afternoon")
         .when(F.col("Hour").between(18, 23), "Evening")
         .otherwise("Night"))

    .withColumn("Vict_Age_Clean", F.col("Vict_Age_Clean").cast("double"))
)

ml_features.write.format("delta").mode("overwrite").saveAsTable("gold_ml_features")
print(f"✓ gold_ml_features saved — {ml_features.count():,} rows")
display(ml_features.limit(10))

✓ gold_ml_features saved — 62,105 rows


AREA,AREA_NAME,Hour,Month,IsWeekend,Reporting_Delay,Has_Weapon,Vict_Age_Clean,Vict_Sex_Clean,Crm_Cd_Desc,Premis_Desc,TimeBucket
11,Northeast,15,8,0,1,0,0.0,Unknown,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),STREET,Afternoon
10,West Valley,18,9,0,1,0,49.0,Male,BURGLARY FROM VEHICLE,GARAGE/CARPORT,Evening
21,Topanga,18,12,0,2,0,37.0,Male,BURGLARY FROM VEHICLE,PARKING LOT,Evening
3,Southwest,1,12,1,2,0,23.0,Unknown,TRESPASSING,STREET,Night
14,Pacific,22,8,0,1,0,36.0,Male,"VANDALISM - FELONY ($400 & OVER, ALL CHURCH VANDALISMS)",STREET,Evening
17,Devonshire,17,8,0,1,0,0.0,Unknown,VEHICLE - STOLEN,STREET,Afternoon
3,Southwest,2,6,0,1,0,23.0,Male,THEFT PLAIN - PETTY ($950 & UNDER),"VEHICLE, PASSENGER/TRUCK",Night
17,Devonshire,14,12,0,1,0,37.0,Male,THEFT PLAIN - PETTY ($950 & UNDER),SINGLE FAMILY DWELLING,Afternoon
12,77th Street,20,12,0,1,0,0.0,Unknown,VEHICLE - STOLEN,STREET,Evening
8,West LA,21,5,1,0,0,0.0,Unknown,VEHICLE - STOLEN,STREET,Evening


# ─────────────────────────────────────────────────────────────────────
# Gold Layer Summary
# ─────────────────────────────────────────────────────────────────────

In [0]:
tables = [
    "gold_area_crime_summary",
    "gold_monthly_trends",
    "gold_hourly_patterns",
    "gold_victim_profiles",
    "gold_weapon_area_analysis",
    "gold_ml_features",
]

print("=" * 60)
print(f"{'Table':<35} {'Rows':>10}  {'Cols':>5}")
print("-" * 60)
for t in tables:
    tdf = spark.table(t)
    print(f"{t:<35} {tdf.count():>10,}  {len(tdf.columns):>5}")
print("=" * 60)
print("\nGold layer complete.")

Table                                     Rows   Cols
------------------------------------------------------------
gold_area_crime_summary                     21     13
gold_monthly_trends                         13      8
gold_hourly_patterns                        48      6
gold_victim_profiles                        26     11
gold_weapon_area_analysis                  141      5
gold_ml_features                        62,105     12

Gold layer complete.
